In [10]:
import socket
import os
import threading

In [11]:
s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
s.bind(('localhost', 9999))
s.listen(5)

In [12]:
clients = []
banned_words = ["bad", "stupid", "idiot"]
allowed_files = [".txt", ".jpg", ".pdf"]

In [13]:
def is_msg_clean(msg):
    for word in banned_words:
        if word in msg.lower():
            return False
    return True

In [14]:
def is_file_allowed(filename):
    file = os.path.splitext(filename)[1]
    return file in allowed_files

In [15]:
def send_to_all_clients(msg):
    for c in clients:
        c.send(msg.encode())

In [16]:
def handle_client(client):
    while True:
        try:
            msg = client.recv(1024).decode()
            if (msg.startswith("file:")):
                filename = msg[5:]
                if is_file_allowed(filename):
                    client.send("File allowed".encode())

                    with open("res_" + filename, "wb") as f:
                        data = client.recv(1024)
                        f.write(data)
                    send_to_all_clients("File received")
                else:
                    client.send("File not allowed".encode())
            else:
                if is_msg_clean(msg):
                    print("Received message:", msg)
                    send_to_all_clients("Message received")
                else:
                    client.send("Message contains banned words".encode())
        except:
            clients.remove(client)
            client.close()
            break

In [ ]:
while True:
    c, addr = s.accept()
    print("Connected:", addr)

    clients.append(c)

    thread = threading.Thread(target=handle_client, args=(c,))
    thread.start()

Connected: ('127.0.0.1', 56663)
Received message: this a message
